# Healthy vs. unhealthy banana dataset counts

This notebook counts:
- healthy and unhealthy banana **images**
- YOLO annotation objects for class `0` (healthy) and class `1` (unhealthy)
- missing, empty, and malformed label files.

In [2]:
from pathlib import Path
from collections import Counter

# Works when Jupyter starts in either the repository root or notebooks/.
DATASET_DIR = next(
    (path for path in [Path.cwd() / 'detection_dataset', Path.cwd().parent / 'detection_dataset'] if path.exists()),
    None,
)
if DATASET_DIR is None:
    raise FileNotFoundError('Could not find detection_dataset. Update DATASET_DIR to its location.')

CLASS_NAMES = {0: 'healthy', 1: 'unhealthy'}
SPLITS = ('train', 'val', 'test')
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

print(f'Dataset: {DATASET_DIR.resolve()}')

Dataset: /home/mehedinaeem/Desktop/Code/Bitol_Computer_Vision_System/detection_dataset


In [3]:
def inspect_split(split):
    image_dir = DATASET_DIR / 'images' / split
    label_dir = DATASET_DIR / 'labels' / split
    images = sorted(p for p in image_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
    label_files = sorted(p for p in label_dir.glob('*.txt') if p.name != 'classes.txt')

    # Image category is inferred from the filename, e.g. healthy_0001.jpg.
    image_counts = Counter()
    for image in images:
        lower_name = image.stem.lower()
        if lower_name.startswith('unhealthy'):
            image_counts[1] += 1
        elif lower_name.startswith('healthy'):
            image_counts[0] += 1
        else:
            image_counts['unknown'] += 1

    annotation_counts = Counter()
    malformed_lines = []
    empty_labels = 0
    for label_file in label_files:
        lines = [line.strip() for line in label_file.read_text(encoding='utf-8').splitlines() if line.strip()]
        if not lines:
            empty_labels += 1
        for line_number, line in enumerate(lines, start=1):
            parts = line.split()
            try:
                class_id = int(parts[0])
                if len(parts) != 5:
                    raise ValueError
                [float(value) for value in parts[1:]]
            except (ValueError, IndexError):
                malformed_lines.append((label_file.name, line_number, line))
                continue
            annotation_counts[class_id] += 1

    image_stems = {p.stem for p in images}
    label_stems = {p.stem for p in label_files}
    return {
        'split': split,
        'healthy_images': image_counts[0],
        'unhealthy_images': image_counts[1],
        'unknown_images': image_counts['unknown'],
        'total_images': len(images),
        'class_0_annotations': annotation_counts[0],
        'class_1_annotations': annotation_counts[1],
        'other_class_annotations': sum(v for k, v in annotation_counts.items() if k not in CLASS_NAMES),
        'total_annotations': sum(annotation_counts.values()),
        'label_files': len(label_files),
        'empty_label_files': empty_labels,
        'images_without_labels': len(image_stems - label_stems),
        'labels_without_images': len(label_stems - image_stems),
        'malformed_lines': malformed_lines,
    }

results = [inspect_split(split) for split in SPLITS]

In [4]:
# Main count table (no third-party packages required).
columns = [
    'split', 'healthy_images', 'unhealthy_images', 'total_images',
    'class_0_annotations', 'class_1_annotations', 'total_annotations', 'label_files'
]
rows = [{column: result[column] for column in columns} for result in results]
rows.append({
    column: ('TOTAL' if column == 'split' else sum(row[column] for row in rows))
    for column in columns
})

widths = {column: max(len(column), *(len(str(row[column])) for row in rows)) for column in columns}
print(' | '.join(column.ljust(widths[column]) for column in columns))
print('-+-'.join('-' * widths[column] for column in columns))
for row in rows:
    print(' | '.join(str(row[column]).ljust(widths[column]) for column in columns))

split | healthy_images | unhealthy_images | total_images | class_0_annotations | class_1_annotations | total_annotations | label_files
------+----------------+------------------+--------------+---------------------+---------------------+-------------------+------------
train | 986            | 712              | 1698         | 1056                | 629                 | 1685              | 814        
val   | 282            | 204              | 486          | 877                 | 561                 | 1438              | 486        
test  | 143            | 105              | 248          | 100                 | 109                 | 209               | 92         
TOTAL | 1411           | 1021             | 2432         | 2033                | 1299                | 3332              | 1392       


In [5]:
# Data-quality checks. All values should normally be zero.
for result in results:
    print(f"\n{result['split'].upper()}")
    print('  unknown image names:  ', result['unknown_images'])
    print('  empty label files:    ', result['empty_label_files'])
    print('  images without label: ', result['images_without_labels'])
    print('  labels without image: ', result['labels_without_images'])
    print('  other class objects:  ', result['other_class_annotations'])
    print('  malformed label lines:', len(result['malformed_lines']))
    if result['malformed_lines']:
        print('  examples:', result['malformed_lines'][:5])


TRAIN
  unknown image names:   0
  empty label files:     0
  images without label:  884
  labels without image:  0
  other class objects:   0
  malformed label lines: 0

VAL
  unknown image names:   0
  empty label files:     1
  images without label:  0
  labels without image:  0
  other class objects:   0
  malformed label lines: 0

TEST
  unknown image names:   0
  empty label files:     0
  images without label:  156
  labels without image:  0
  other class objects:   0
  malformed label lines: 0
